In [6]:
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


# Block 4: Wavelength, weather, and the scan
### How does wavelength shape what you see, and how do you assemble a volume?

Most radar networks across Africa run C-band (some X-band), and a shorter wavelength *attenuates*. This block has three widgets. Run the setup cell once, then work through the widgets in order, since each builds on the last.

Each widget's questions come in three levels. Start at whichever level feels comfortable and push as far as you can — they're meant to stretch, not to be finished. Time per widget is short, so don't expect to clear every level here; the more advanced questions are designed to be picked up again after the workshop.

## Before we start
This block is the synthesis — wavelength effects you can't avoid, and how every earlier idea stacks into one volume scan. Three things to hold onto.

1. **Shorter wavelength, bigger price.** A shorter wavelength buys a finer beam from a smaller dish (Block 1) — but rain absorbs and scatters it more, so the beam is attenuated. Negligible at S-band, real at C-band, severe at X-band.

2. **Sensitivity and attenuation pull opposite ways.** At the same dish, a shorter wavelength is intrinsically *more* sensitive — it can detect weaker echoes — but it also attenuates more, and that penalty grows with range. Which band "sees" a faint distant storm is a competition between the two.

3. **A scan is many beams, and time is finite.** Building a volume trades vertical detail, clean velocity (Block 3), and update speed against each other — the problem phased arrays exist to solve.

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

C = 2.998e8; A_E = 6371.0
kB, T0 = 1.380649e-23, 290.0
ATTEN = {"S": (0.0001, 1.0), "C": (0.006, 1.0), "X": (0.03, 1.0)}  # k = a*R^b, dB/km one-way

def rain_scene(x_km, peak=80, bg=2, centre=40, width=8):
    return bg + peak*np.exp(-((x_km - centre)/width)**2)
def z_from_rain(R):
    return 10*np.log10(200 * np.maximum(R, 1e-6)**1.6)
def path_attenuation(x_km, R, band):
    a, b = ATTEN[band]; return 2*np.cumsum(a * R**b * np.gradient(x_km))
def beam_height(r_km, elev_deg):
    Re = (4/3)*A_E; phi = np.radians(elev_deg)
    return np.sqrt(r_km**2 + Re**2 + 2*r_km*Re*np.sin(phi)) - Re
def min_Z(r_km, power_kW=750, dish_m=8.5, freq_ghz=2.8, pulse_us=1.57,
          eta=0.45, NF_dB=4, Kw2=0.93, snr_dB=3):
    # minimum detectable reflectivity (dBZ) vs range; carries the 1/λ⁴ sensitivity at fixed dish
    lam = C/(freq_ghz*1e9); tau = pulse_us*1e-6; r = np.asarray(r_km, float)*1e3
    G = eta*(np.pi*dish_m/lam)**2; theta = 1.27*lam/dish_m
    N = kB*T0*(1/tau)*10**(NF_dB/10); snr = 10**(snr_dB/10)
    K = (np.pi**3 * C * power_kW*1e3 * G**2 * theta**2 * tau * Kw2)/(1024*np.log(2)*lam**2)
    return 10*np.log10((snr*N*r**2/K)/1e-18)

VCP = {"VCP 21 (9 tilts)":  [0.5,1.45,2.4,3.35,4.3,6.0,9.9,14.6,19.5],
       "VCP 12 (14 tilts)": [0.5,0.9,1.3,1.8,2.4,3.1,4.0,5.1,6.4,8.0,10.0,12.5,15.6,19.5],
       "VCP 31 clear-air (5)": [0.5,1.5,2.5,3.5,4.5],
       "fast low-level (3)": [0.5,1.5,2.5]}

print("Block 4 ready | attenuation across cell (dB):",
      {b: round(float(path_attenuation(np.linspace(0,120,600), rain_scene(np.linspace(0,120,600)), b)[-1]),1) for b in "SCX"},
      "| intrinsic MDS @100 km S/C/X:", [round(float(min_Z(100,freq_ghz=f)),1) for f in (2.8,5.6,9.4)])

Block 4 ready | attenuation across cell (dB): {'S': 0.3, 'C': 16.5, 'X': 82.5} | intrinsic MDS @100 km S/C/X: [-2.9, -15.0, -24.0]


## 1 — Shorter wavelength, bigger price: attenuation

A beam crosses a rain cell. Rain doesn't just reflect energy, it absorbs and scatters it *out* of the beam, so everything behind the cell is seen through a curtain of rain. That loss is negligible at S-band, real at C-band, and can wipe out the echo entirely at X-band.

In [9]:
def plot_attenuation(peak_mmh=80):
    x = np.linspace(0, 120, 600); R = rain_scene(x, peak_mmh); z_true = z_from_rain(R)
    plt.figure(figsize=(8, 4.6))
    plt.plot(x, z_true, "k--", label="true reflectivity")
    for band, col in [("S", "tab:blue"), ("C", "tab:orange"), ("X", "tab:red")]:
        plt.plot(x, z_true - path_attenuation(x, R, band), color=col, lw=2, label=band + "-band observed")
    plt.ylim(-10, 60); plt.grid(alpha=0.3); plt.legend()
    plt.xlabel("range along beam (km)"); plt.ylabel("reflectivity (dBZ)")
    plt.title("attenuation carves a shadow behind the cell")
    plt.show()

interact(plot_attenuation,
         peak_mmh=FloatSlider(value=80, min=10, max=150, step=5, description="peak mm/h"));

interactive(children=(FloatSlider(value=80.0, description='peak mm/h', max=150.0, min=10.0, step=5.0), Output(…

**Basic**

1. Which band reads the cell peak closest to the truth? Which loses everything behind the core?
2. At 80 mm/h, roughly how many dBZ does the C-band echo lose behind the cell?

**A little further**

1. Most networks across Africa are C-band. What would that shadow do to a rainfall estimate further downrange of a heavy core?
2. Raise the peak rain rate. Does the X-band shadow deepen steadily, or accelerate? What does that imply for heavy convection?

**Going deeper**

1. Two-way attenuation accumulates along the path, so the shadow depends on everything the beam has already crossed. Raise the cell intensity and watch the behind-cell deficit grow faster than linearly — why (it's a path integral of specific attenuation)? Why is a second cell *behind* the first essentially invisible at X-band?
2. At S-band the shadow is negligible — which is why NEXRAD needs no attenuation correction but African C-band networks do. Read the two-way C-band loss across a heavy cell off the widget, and convert it to the rainfall underestimate downrange (use Z = 200·R^1.6).

## 2 — Sensitivity vs attenuation: which band sees the distant storm?

Two wavelength effects pull in opposite directions. At the same dish, a *shorter* wavelength is intrinsically more sensitive — it can detect weaker echoes (the 1/λ⁴ in the radar equation, **dotted** lines). But a shorter wavelength also attenuates more, and that penalty *grows with range* (**solid** lines). Slide the rain the beam passes through and watch the curves cross: the band that sees a faint distant storm best is whichever wins this competition.

In [11]:
def plot_detectability(path_rain_mmh=5):
    r = np.linspace(5, 250, 400)
    eff = {}
    plt.figure(figsize=(8, 4.8))
    for name, f, col in [("S", 2.8, "tab:blue"), ("C", 5.6, "tab:orange"), ("X", 9.4, "tab:red")]:
        a, b = ATTEN[name]
        intrinsic = min_Z(r, freq_ghz=f)                 # 1/λ⁴ sensitivity at the same dish
        eff[name] = intrinsic + 2*a*path_rain_mmh**b*r    # plus two-way path attenuation (dB)
        plt.plot(r, intrinsic, color=col, ls=":", lw=1, alpha=0.45)
        plt.plot(r, eff[name], color=col, lw=2.2, label=name + "-band")
    idx = np.where(np.diff(np.sign(eff["X"] - eff["S"])) > 0)[0]
    cross = r[idx[0]] if len(idx) else None
    if cross is not None:
        plt.axvline(cross, color="gray", ls="--", lw=0.8)
        ttl = "through " + str(path_rain_mmh) + " mm/h rain: X-band beats S-band only inside " + str(round(cross)) + " km"
    else:
        ttl = "no intervening rain: shorter wavelength wins at every range (1/λ⁴)"
    plt.ylim(-30, 50); plt.xlim(0, 250); plt.grid(alpha=0.3); plt.legend(loc="lower right")
    plt.xlabel("range (km)"); plt.ylabel("minimum detectable Z (dBZ)")
    plt.text(8, 44, "dotted: no-rain sensitivity · solid: through the rain", fontsize=8, color="gray")
    plt.title(ttl)
    plt.show()

interact(plot_detectability,
         path_rain_mmh=FloatSlider(value=5, min=0, max=20, step=1, description="path rain mm/h"));

interactive(children=(FloatSlider(value=5.0, description='path rain mm/h', max=20.0, step=1.0), Output()), _do…

**Basic**

1. Set the rain to 0. Which band detects the weakest echoes at every range, and why (think 1/λ⁴ at the same dish)?
2. Now raise the rain. Which band's curve climbs fastest, and which barely moves?

**A little further**

1. At a moderate rain rate, read the range where the X-band curve crosses the S-band curve. Inside it, which band is better; beyond it, which? Put the competition in one sentence.
2. Push the rain rate higher. Does the crossover move toward or away from the radar? Why does heavier rain shrink X-band's useful sensitivity range?

**Going deeper**

1. The intrinsic advantage (dotted) is a fixed ~21 dB for X over S at the same dish — the same at every range — while the attenuation penalty (the gap from dotted to solid) grows *with* range, because it's a path integral. Explain why a fixed head-start always loses to a penalty that accumulates, so for distant weak rain the longer wavelength wins eventually no matter how big the head-start.
2. Tie it to widget 1 and the African context: a compact C-band radar is more sensitive than NEXRAD at the same dish, yet it sees *less* of a distant storm through heavy rain. For a network choosing a band, what does this trade say — and how could polarimetric attenuation correction (KDP) shift where the crossover sits?

## 3 — Putting it together: scan strategies

A volume coverage pattern is a stack of tilts. The fan (on Block 1's height axes) exposes the three things a VCP trades: the **cone of silence** above the top tilt, the **low-level gap** below the bottom one, and the **update time** that grows with every tilt — the exact problem electronically-steered phased arrays exist to solve.

In [12]:
def plot_vcp(pattern="VCP 21 (9 tilts)", rpm=3.0):
    tilts = VCP[pattern]; top, bottom = max(tilts), min(tilts); r = np.linspace(1, 300, 300)
    plt.figure(figsize=(8, 5.2))
    for e in tilts: plt.plot(r, beam_height(r, e), lw=1, color="tab:blue")
    plt.fill_between(r, beam_height(r, top), 20, color="tab:red", alpha=0.06)
    plt.text(35, 16.5, "cone of silence", color="tab:red", fontsize=9)
    plt.fill_between(r, 0, beam_height(r, bottom), color="gray", alpha=0.10)
    plt.text(150, 0.4, "low-level gap", color="dimgray", fontsize=9)
    vol = len(tilts)*60/rpm
    plt.xlim(0, 300); plt.ylim(0, 20); plt.grid(alpha=0.3)
    plt.xlabel("range (km)"); plt.ylabel("height (km)")
    plt.title(pattern + ": volume time " + str(round(vol)) + " s (" + str(round(vol/60,1))
              + " min)  |  cone half-angle " + str(round(90-top)) + "°")
    plt.show()

interact(plot_vcp,
         pattern=Dropdown(options=list(VCP), value="VCP 21 (9 tilts)", description="pattern"),
         rpm=FloatSlider(value=3, min=1, max=6, step=0.5, description="rpm"));

interactive(children=(Dropdown(description='pattern', options=('VCP 21 (9 tilts)', 'VCP 12 (14 tilts)', 'VCP 3…

**Basic**

1. Read the volume update time for VCP 21 and VCP 12 at 3 rpm. Which is faster, and why?
2. Find the cone of silence: at 30 km range, how high can a storm top be before it sits above the top tilt and is missed?

**A little further**

1. Choose a pattern that updates in under 3 minutes at 3 rpm. How many tilts can you afford, and what vertical coverage do you sacrifice?
2. Connect to Block 3: cleaner velocity needs more pulses per radial, so suppose each tilt takes 25 s instead of 20. Recompute the VCP 12 volume time — and name the technology this trade motivates.

**Going deeper**

1. Tie all three blocks together: more tilts buys vertical coverage but slows the volume (this widget); more pulses per radial buys cleaner velocity (Block 3) but slows each tilt; and the lowest tilt still overshoots low-level weather at range (Block 1). Build a VCP you'd actually run for a fast-moving severe line, and justify every trade.
2. The cone of silence has half-angle 90°−(top tilt). Compute it for the top tilt, then find the range at which that top tilt's beam reaches 12 km (a storm top). Between the radar and that range, where are you blind to tops — and why does this argue for a higher top tilt or a faster-revisiting phased array?

## 4 — Radar data

*Placeholder — drop the chosen image in the cell below (replace the link with your uploaded `block4_image1.jpg`). **Important:** this block's attenuation story needs **C-band (or X-band) dual-pol** data — NEXRAD's S-band barely attenuates, so it cannot show the shadow. See the accompanying notes for sources (a C-band case is essential here).*

The panel shows reflectivity and differential phase (ΦDP) through a strong convective cell observed by a **C-band dual-pol radar** — the wavelength most African networks run. ΦDP is the operational tool that addresses the attenuation you explored in widgets 1 and 2.

**Basic**

Behind the strongest cell, does the reflectivity drop into a "shadow" — lower than the surrounding cells would lead you to expect? At C-band that shadow is attenuation, not a real dry slot. How could you confirm it's attenuation rather than an actual gap in the rain?

**A little further**

Follow ΦDP (differential phase) along a radial through the core: it ramps up across the cell and stays high behind it. Why does that ramp measure how much attenuation the beam has suffered, and how is it used to add the lost reflectivity back (KDP / ΦDP-based attenuation correction)?

**Going deeper**

Compare raw and attenuation-corrected Z behind the core, and check ZDR (it reads too low behind the cell — differential attenuation). Explain why a KDP-based rain estimate downrange is trustworthy while Z→R and ZDR→R are both biased low — and why this is exactly why dual-pol matters most for the C-band networks across Africa. If you only had S-band (NEXRAD) data, what would this shadow look like, and why?

![Image](https://raw.githubusercontent.com/sebastiantorr/Clouds4Africa/main/content/block4_image1.jpg)

## Block 4 takeaway
**A shorter wavelength buys a finer beam and more intrinsic sensitivity from a smaller dish, but pays in attenuation that grows with range — so which band sees a distant storm is a competition between the two, and assembling a volume adds a further trade between vertical detail, clean velocity, and update speed.**